# Fair Price Checker: Exploratory Data Analysis (EDA)

This notebook inspects the CarDekho dataset, performs data cleaning, explores feature relationships, and visualizes price dynamics to inform the regression modeling pipeline.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
df = pd.read_csv('../data/cardekho.csv')
df = df.drop(columns=[c for c in df.columns if c.startswith('Unnamed')], errors='ignore')
print('Dataset Shape:', df.shape)
df.info()

## 1. Target Variable: Selling Price Distribution
Used car prices in India exhibit extreme right skewness. Applying a `log1p` transformation stabilizes variance and normalizes errors for linear and tree models.

In [2]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df['selling_price'] / 100000, kde=True, ax=axes[0], color='#2563eb')
axes[0].set_title('Raw Price Distribution (Lakhs INR)')

df['log_price'] = np.log1p(df['selling_price'])
sns.histplot(df['log_price'], kde=True, ax=axes[1], color='#16a34a')
axes[1].set_title('Log-Transformed Price Distribution')
plt.tight_layout()
plt.show()

**Key takeaway:** Raw prices are heavily right-skewed. The log transform makes the target distribution substantially closer to a Gaussian bell curve, preventing high-value luxury cars from disproportionately penalizing model weights during MSE loss minimization.

## 2. Vehicle Age vs. Selling Price
Depreciation curve of used vehicles over time.

In [3]:
plt.figure(figsize=(8, 5))
sns.lineplot(data=df, x='vehicle_age', y='selling_price', estimator='mean', color='#2563eb')
plt.title('Vehicle Age vs. Average Selling Price')
plt.xlabel('Vehicle Age (Years)')
plt.ylabel('Selling Price (INR)')
plt.show()

**Key takeaway:** Depreciation is non-linear: the sharpest price decline occurs between years 0 and 5, after which the curve flattens. This strongly supports using ensemble tree models (like Random Forest) over pure Linear Regression.

## 3. Correlation Heatmap
Analysis of collinearity among numerical features.

In [4]:
num_cols = ['vehicle_age', 'km_driven', 'mileage', 'engine', 'max_power', 'seats', 'selling_price', 'log_price']
plt.figure(figsize=(9, 7))
sns.heatmap(df[num_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Feature Correlation Matrix')
plt.show()

**Key takeaway:** `max_power` and `engine` display strong positive correlation with price (r = 0.75+), while `vehicle_age` has strong negative correlation (r = -0.59). Engine and max_power have correlation r = 0.78, which is just below the 0.85 threshold, making both informative for tree algorithms.